In [0]:
%sql
-- notebook source
-- Bronze — Ingestão Raw
-- Ingestão do arquivo da ANS **"as-is"**: mesma estrutura de colunas do CSV
-- original (tudo como STRING), sem nenhuma transformação de negócio.
-- Só adicionamos metadados técnicos de ingestão para rastreabilidade
-- (auditoria / linhagem).
-- Fonte: https://dadosabertos.ans.gov.br/FTP/PDA/informacoes_consolidadas_de_beneficiarios-024/202508/pda-024-icb-TO-2025_08.zip
-- Usamos `COPY INTO`, que é idempotente (não reprocessa arquivo já carregado) — essencial para o job diário não duplicar dados quando não houver arquivo novo no dia.

In [0]:
%sql
-- utilizando o catalogo e schema 

USE CATALOG bmg_saude_desafio;
USE SCHEMA bronze;

In [0]:
%sql
-- criando a tabela de beneficiarios

CREATE TABLE IF NOT EXISTS bronze.beneficiarios_ans_raw (
  ID_CMPT_MOVEL             STRING,
  CD_OPERADORA               STRING,
  NM_RAZAO_SOCIAL            STRING,
  NR_CNPJ                    STRING,
  MODALIDADE_OPERADORA       STRING,
  SG_UF                      STRING,
  CD_MUNICIPIO                STRING,
  NM_MUNICIPIO                STRING,
  TP_SEXO                    STRING,
  DE_FAIXA_ETARIA            STRING,
  DE_FAIXA_ETARIA_REAJ       STRING,
  CD_PLANO                   STRING,
  TP_VIGENCIA_PLANO          STRING,
  DE_CONTRATACAO_PLANO       STRING,
  DE_SEGMENTACAO_PLANO       STRING,
  DE_ABRG_GEOGRAFICA_PLANO   STRING,
  COBERTURA_ASSIST_PLAN      STRING,
  TIPO_VINCULO                STRING,
  QT_BENEFICIARIO_ATIVO      STRING,
  QT_BENEFICIARIO_ADERIDO    STRING,
  QT_BENEFICIARIO_CANCELADO  STRING,
  DT_CARGA                   STRING,
  -- metadados técnicos de ingestão
  _source_file                STRING,
  _ingested_at                 TIMESTAMP
)
USING DELTA
COMMENT 'Bronze - dados brutos de beneficiarios ANS (as-is, tudo STRING)';

In [0]:
%sql

-- Ingestão idempotente via COPY INTO
-- O arquivo é lido a partir da landing zone (Volume do Unity Catalog).
-- Em produção, um job upstream (ou o próprio Autoloader monitorando a pasta) deposita o CSV do mês ali antes deste passo rodar.

COPY INTO bronze.beneficiarios_ans_raw
FROM (
  SELECT
    ID_CMPT_MOVEL, CD_OPERADORA, NM_RAZAO_SOCIAL, NR_CNPJ, MODALIDADE_OPERADORA,
    SG_UF, CD_MUNICIPIO, NM_MUNICIPIO, TP_SEXO, DE_FAIXA_ETARIA, DE_FAIXA_ETARIA_REAJ,
    CD_PLANO, TP_VIGENCIA_PLANO, DE_CONTRATACAO_PLANO, DE_SEGMENTACAO_PLANO,
    DE_ABRG_GEOGRAFICA_PLANO, COBERTURA_ASSIST_PLAN, TIPO_VINCULO,
    QT_BENEFICIARIO_ATIVO, QT_BENEFICIARIO_ADERIDO, QT_BENEFICIARIO_CANCELADO, DT_CARGA
  FROM '/Volumes/bmg_saude_desafio/bronze/landing_zone/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header'    = 'true',
  'delimiter' = ';',
  'quote'     = '"',
  'encoding'  = 'UTF-8'
)
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
-- Preenche os metadados técnicos apenas para as linhas recém-carregadas que ainda não têm _ingested_at.

UPDATE bronze.beneficiarios_ans_raw
SET _source_file = 'pda-024-icb-TO-2025_08.csv',
    _ingested_at  = current_timestamp()
WHERE _ingested_at IS NULL;

In [0]:
%sql

-- Checagem rápida de volumetria

SELECT
  COUNT(*)                    AS total_linhas,
  COUNT(DISTINCT DT_CARGA)    AS competencias_distintas,
  MIN(_ingested_at)           AS primeira_carga,
  MAX(_ingested_at)           AS ultima_carga
FROM bronze.beneficiarios_ans_raw;